# Smart Lender – AI-Powered Loan Approval Prediction System

This notebook covers the step-by-step development of a machine learning workflow for retail loan eligibility prediction. We will explore the dataset, resolve missing values, build preprocessing pipelines, train and benchmark classification models, and serialize the champion model.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report, roc_curve

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier

%matplotlib inline
sns.set_theme(style="whitegrid")

## 1. Load Dataset

We load the downloaded or generated dataset (`dataset/loan_data.csv`).

In [ ]:
# Read data
df = pd.read_csv('../dataset/loan_data.csv')
print(f"Dataset Shape: {df.shape}")
df.head()

## 2. Exploratory Data Analysis (EDA)

Let's inspect class balance, correlations, distributions, and null structures.

In [ ]:
# Print basic info
df.info()

In [ ]:
# Visualizing missing values
plt.figure(figsize=(10, 6))
sns.heatmap(df.isnull(), cbar=False, cmap='viridis')
plt.title("Null Values Heatmap")
plt.show()

In [ ]:
# Class distributions for Loan Status
plt.figure(figsize=(6, 4))
sns.countplot(x='Loan_Status', data=df, palette='Set2')
plt.title('Target Variable Distribution')
plt.show()

In [ ]:
# Numerical feature relations vs Target
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(x='Loan_Status', y='ApplicantIncome', data=df, ax=axes[0], palette='Set3')
axes[0].set_title('Applicant Income vs Loan Status')
axes[0].set_yscale('log')

sns.boxplot(x='Loan_Status', y='LoanAmount', data=df, ax=axes[1], palette='Set3')
axes[1].set_title('Loan Amount vs Loan Status')
plt.show()

## 3. Data Preprocessing & Splitting

We separate features, label encode our target variable, and partition the dataset.

In [ ]:
X = df.drop(columns=['Loan_ID', 'Loan_Status'])
y = df['Loan_Status'].map({'Y': 1, 'N': 0})

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

num_features = ['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term']
cat_features = ['Gender', 'Married', 'Dependents', 'Education', 'Self_Employed', 'Credit_History', 'Property_Area']

In [ ]:
# Define ColumnTransformer pipelines
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', drop='if_binary'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', num_transformer, num_features),
    ('cat', cat_transformer, cat_features)
])

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

## 4. Train & Benchmark Models

We train and score: Decision Tree, Random Forest, K-Nearest Neighbors, and XGBoost.

In [ ]:
models = {
    'Decision Tree': DecisionTreeClassifier(random_state=42, max_depth=5),
    'Random Forest': RandomForestClassifier(random_state=42, n_estimators=100, max_depth=6),
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=5),
    'XGBoost': XGBClassifier(random_state=42, n_estimators=100, max_depth=4, learning_rate=0.05, eval_metric='logloss')
}

results = []
trained_models = {}
plt.figure(figsize=(10, 8))

for name, model in models.items():
    model.fit(X_train_processed, y_train)
    trained_models[name] = model
    
    preds = model.predict(X_test_processed)
    probs = model.predict_proba(X_test_processed)[:, 1]
    
    acc = accuracy_score(y_test, preds)
    prec = precision_score(y_test, preds)
    rec = recall_score(y_test, preds)
    f1 = f1_score(y_test, preds)
    auc = roc_auc_score(y_test, probs)
    
    results.append({
        'Model': name,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1 Score': f1,
        'ROC-AUC': auc
    })
    
    fpr, tpr, _ = roc_curve(y_test, probs)
    plt.plot(fpr, tpr, label=f"{name} (AUC = {auc:.3f})")

plt.plot([0, 1], [0, 1], 'k--', label="Random Guess")
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves Comparison')
plt.legend(loc='lower right')
plt.grid(True)
plt.show()

In [ ]:
results_df = pd.DataFrame(results)
results_df

## 5. Model Serialization

Let's select the champion model automatically and write model objects to pickling format.

In [ ]:
best_idx = results_df['Accuracy'].idxmax()
best_model_name = results_df.loc[best_idx, 'Model']
print(f"Best model is: {best_model_name}")

# Export files
with open('../model/best_model.pkl', 'wb') as f:
    pickle.dump(trained_models[best_model_name], f)

with open('../model/preprocessor.pkl', 'wb') as f:
    pickle.dump(preprocessor, f)
    
print("Serialized best_model.pkl and preprocessor.pkl successfully.")